# Day 7：请求生命周期与调度地图

配套 [Day 7讲义](../docs/DAY07_SCHEDULER_MAP.md) 与 [学习记录](../docs/DAY07_STUDY_LOG.md)。
今天只做离散调度和地址映射教学实验，不部署真实vLLM，不测GPU时间。
记住：round不是毫秒，Paged KV不是scheduler，continuous batching也不是自动加速。

## 先画状态机

请求：`waiting → prefill → running decode → finished/released`。
取消、异常或容量不足可能进入cancelled/failed分支。
scheduler决定本轮谁运行；KV manager负责物理页；Attention kernel负责实际数值计算。
先预测A/B/C三种策略的first_round和finish_round，再执行。

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/mini_vllm/scheduler.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from learning.mini_vllm.scheduler import Job, simulate
jobs = [Job('A', tokens=3, arrival=0), Job('B', tokens=1, arrival=0), Job('C', tokens=2, arrival=1)]
slots, token_budget = 2, 2
print('jobs:', jobs)
print('slots=', slots, 'token_budget=', token_budget)

## 1. 对比三种调度策略

Job.tokens是剩余decode工作项，本实验省略prefill、首token采样、真实模型与KV容量。
- sequential：同一时刻只服务一个请求。
- static：初始运行集合不因单个请求完成而补入新请求，整批空后再装入。
- continuous：每轮边界都可以用已释放的槽位接纳等待请求。

预测：C在哪种模式下更早执行？这只是逻辑轨迹，不是性能测试。
round从0编号；first_round/finish_round是执行结束时的计数，即对应round+1。
first_round不是独立的准入事件时间；本例预算足够，接纳后当轮就执行。

In [ ]:
reports = {}
for mode in ('sequential', 'static', 'continuous'):
    report = simulate(jobs, mode=mode, slots=slots, token_budget=token_budget)
    reports[mode] = report
    print('\n', mode)
    print('first_round:', report['first_round'])
    print('finish_round:', report['finish_round'])
    for row in report['trace']:
        print(f"round {row['round']}: executed={row['executed']}, waiting={row['waiting']}")
assert reports['continuous']['first_round']['C'] == 2
assert reports['static']['first_round']['C'] == 4
assert reports['continuous']['rounds'] == 3
assert reports['static']['rounds'] == 5
assert reports['sequential']['rounds'] == 6
expected = {
    'sequential': [('A',), ('A',), ('A',), ('B',), ('C',), ('C',)],
    'static': [('A', 'B'), ('A',), ('A',), ('C',), ('C',)],
    'continuous': [('A', 'B'), ('A', 'C'), ('A', 'C')],
}
for mode, report in reports.items():
    assert [tuple(row['executed']) for row in report['trace']] == expected[mode]
print('\nC：continuous在round=1执行，first_round=2；static在round=3执行，first_round=4。')
print('这些是离散逻辑计数，不是GPU时间。')

### 槽位和预算不是同一件事

固定slots=2，将token_budget从2改为1。两条请求可留在running，但每轮只执行一条。
先预测：continuous总轮数是否一定比static少？本例6个工作项、每轮1项，三种策略都需6轮。
检查每条请求工作总量、到达前不执行、每轮不重复，以及first/finish计数对应trace。

In [ ]:
for mode in ('sequential', 'static', 'continuous'):
    limited = simulate(jobs, mode=mode, slots=2, token_budget=1)
    assert limited['rounds'] == 6
    for job in jobs:
        execution_rounds = [row['round'] for row in limited['trace'] if job.name in row['executed']]
        assert len(execution_rounds) == job.tokens
        assert min(execution_rounds) >= job.arrival
        assert limited['first_round'][job.name] == execution_rounds[0] + 1
        assert limited['finish_round'][job.name] == execution_rounds[-1] + 1
    for row in limited['trace']:
        assert len(row['executed']) <= 1
        assert len(row['executed']) == len(set(row['executed']))
    print(mode, 'budget=1, rounds=', limited['rounds'], 'trace=',
          [row['executed'] for row in limited['trace']])
print('预算改变会影响结果；continuous不是所有设定下都减少总轮数。')

## 2. Paged KV：逻辑位置到物理页

假设`block_size=4`。逻辑位置6属于第`6//4=1`个逻辑block，在块内偏移`6%4=2`。
请求自己的block table再把逻辑block 1翻译成物理block，例如物理块7。
这解决的是KV分配/映射，不负责决定本轮哪个请求运行。

In [ ]:
block_size = 4
logical_position = 6
logical_block = logical_position // block_size
offset = logical_position % block_size
block_table = [3, 7, 1]
physical_block = block_table[logical_block]
print('logical position:', logical_position)
print('logical block:', logical_block)
print('offset:', offset)
print('physical block:', physical_block)
assert (logical_block, offset, physical_block) == (1, 2, 7)
retained_positions = 7
needed_blocks = (retained_positions + block_size - 1) // block_size
unused_slots = needed_blocks * block_size - retained_positions
assert (needed_blocks, unused_slots) == (2, 1)
print('7个有效位置按需分配2块，容量8，页尾仍有1个空槽。')
print('这里只算映射和容量，不分配真实KV、不测显存或PagedAttention kernel。')

## 3. 自测与参考答案

**问：Continuous Batching做了什么？**
答：在decode轮次边界更新运行集合，完成请求释放槽位，等待请求可以补入。

**问：Paged KV和scheduler是一回事吗？**
答：不是。Paged KV管理物理页和映射；scheduler管理准入、预算、完成和等待。

**问：C提前进入running是否证明吞吐更高？**
答：不证明。还没有wall-clock、GPU同步、kernel、通信、prefill成本或SLO数据。

**问：prefill和decode为什么不能混成一件事？**
答：prefill通常一次处理多个prompt token并建立Cache；decode通常每轮追加一个token并读取历史K/V，shape和预算不同。

**问：预算=1时continuous一定更少轮吗？**
答：不一定，本例三种模式都用6轮。slots是运行集合容量，budget是本轮工作额度。

**问：first_round=2为什么trace显示round=1？**
答：前者是首次执行结束计数，后者从0编号；不是两次实际执行或延迟2秒。

**问：模拟器释放了真实KV页吗？**
答：没有，仅从running移除完成Job。实际KV释放需要真实Cache manager；页尾也不保证零浪费。

**个人验收**：先在[Day 7学习记录](../docs/DAY07_STUDY_LOG.md)填写预测，再运行本Notebook。
运行参考实现不自动标记个人掌握。